In [57]:
import pandas as pd
import numpy as np
import ast
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
import json
import joblib
import os
import torch
import warnings
import google.generativeai as genai
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

warnings.filterwarnings('ignore')

In [58]:
data_dir = '../Data'
df_books = pd.read_csv(f'{data_dir}/Books_Details.csv')
df_reviews = pd.read_csv(f'{data_dir}/Final_Reviews.csv')

In [59]:
# ==========================================
# reviews
# ==========================================
df_ratings = df_reviews.rename(columns={
    'Student_Id': 'user_id',
    'Book_Id': 'book_id',
    'review/score': 'rating',
    'review/text': 'review_text'
})[['user_id', 'book_id', 'rating', 'review_text']]

# تحويل التقييمات لأرقام صحيحة وتجاهل أي قيم فارغة
df_ratings.dropna(subset=['user_id', 'book_id', 'rating'], inplace=True)
df_ratings['user_id'] = df_ratings['user_id'].astype(int)
df_ratings['book_id'] = df_ratings['book_id'].astype(int)
df_ratings['rating'] = df_ratings['rating'].astype(float)

In [60]:
# ==========================================
# books
# ==========================================
df = df_books.rename(columns={
    'Book_Id': 'book_id',
    'Title': 'title',
    'categories': 'genre'
})

def clean_list_string(text):
    if pd.isna(text): return ""
    try:
        parsed_list = ast.literal_eval(text)
        if isinstance(parsed_list, list):
            return " ".join(parsed_list)
    except:
        pass
    return str(text).replace("['", "").replace("']", "")

df['genre'] = df['genre'].apply(clean_list_string)
df['authors'] = df['authors'].apply(clean_list_string)
df['description'] = df['description'].fillna("")

df['soup'] = df['title'] + " " + df['authors'] + " " + df['genre'] + " " + df['description']
df['soup'] = df['soup'].str.lower()

In [61]:
print(f"the number of reviews: ({len(df_ratings)})")
print(f"the number of books: ({len(df)})")
print("\nthe head of the ratings data:")
display(df_ratings.head())
print("\nthe head of the books data with soup:")
display(df[['book_id', 'title', 'genre', 'soup']].head())

the number of reviews: (35556)
the number of books: (4971)

the head of the ratings data:


,user_id,book_id,rating,review_text
0,1,1682,5.0,"the title of this book comes from avot 3:2, in..."
1,2,1469,4.0,howard zinn's approach to history comes direct...
2,3,1469,5.0,this is an essential book if a person wants to...
3,4,1469,2.0,i have no background in history and do not pre...
4,5,1517,5.0,"fr. faherty has done it again! ""the st. louis ..."



the head of the books data with soup:


,book_id,title,genre,soup
0,1,"GOD BLESS YOU, MR. ROSEWATER, or Pearls Before...",Absurd (Philosophy),"god bless you, mr. rosewater, or pearls before..."
1,2,Doing Your Masters Dissertation,Academic writing,doing your masters dissertation chris hart aca...
2,3,Accounting (Chapters 1-12),Accounting,accounting (chapters 1-12) jeffrey slater mich...
3,4,Spirit and Matter: New Horizons for Medicine,Alternative medicine,spirit and matter: new horizons for medicine j...
4,5,The Magic School Bus Shows And Tells: A Book A...,Archaeology,the magic school bus shows and tells: a book a...


## **Content Based:**

In [62]:
print("Loading NLP Model and Encoding Books... This might take a minute.")

# 1. Load the multilingual NLP model
nlp_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Generate embeddings for all books based on their 'soup' (title + author + genre + description)
book_embeddings = nlp_model.encode(df['soup'].tolist(), convert_to_tensor=True)

print("✅ Books Encoded Successfully!")

# 3. Convert embeddings to NumPy array (safely handling both Tensor and Array types)
if isinstance(book_embeddings, torch.Tensor):
    embeddings_numpy = book_embeddings.cpu().numpy()
else:
    embeddings_numpy = np.array(book_embeddings)

# 4. Calculate the Semantic Cosine Similarity Matrix
cosine_sim = cosine_similarity(embeddings_numpy, embeddings_numpy)

print(f"📊 Semantic Similarity Matrix Shape: {cosine_sim.shape}")

Loading NLP Model and Encoding Books... This might take a minute.
✅ Books Encoded Successfully!
📊 Semantic Similarity Matrix Shape: (4971, 4971)


In [63]:
df

,book_id,title,description,authors,image,previewLink,infoLink,genre,soup
0,1,"GOD BLESS YOU, MR. ROSEWATER, or Pearls Before...",A lawyer schemes to gain control of a large fo...,Kurt Vonnegut,http://books.google.com/books/content?id=E8TZA...,http://books.google.com/books?id=E8TZAAAAMAAJ&...,http://books.google.com/books?id=E8TZAAAAMAAJ&...,Absurd (Philosophy),"god bless you, mr. rosewater, or pearls before..."
1,2,Doing Your Masters Dissertation,A practical and comprehensive guide to researc...,Chris Hart,http://books.google.com/books/content?id=drmwc...,http://books.google.com/books?id=drmwc95ezaYC&...,http://books.google.com/books?id=drmwc95ezaYC&...,Academic writing,doing your masters dissertation chris hart aca...
2,3,Accounting (Chapters 1-12),"""New discussions of modern accounting techniqu...",Jeffrey Slater Michael Deschamps Mike Deschamps,http://books.google.com/books/content?id=4P81s...,http://books.google.nl/books?id=4P81swEACAAJ&d...,http://books.google.nl/books?id=4P81swEACAAJ&d...,Accounting,accounting (chapters 1-12) jeffrey slater mich...
3,4,Spirit and Matter: New Horizons for Medicine,A radical paradigm for integrating modern West...,José Lacerda de Azevedo,http://books.google.com/books/content?id=5PteP...,http://books.google.com/books?id=5PtePQAACAAJ&...,http://books.google.com/books?id=5PtePQAACAAJ&...,Alternative medicine,spirit and matter: new horizons for medicine j...
4,5,The Magic School Bus Shows And Tells: A Book A...,Ms. Frizzle and the gang travel back in time t...,Joanna Cole Jackie Posner,http://books.google.com/books/content?id=XoBGx...,http://books.google.com/books?id=XoBGxTlkwm4C&...,http://books.google.com/books?id=XoBGxTlkwm4C&...,Archaeology,the magic school bus shows and tells: a book a...
...,...,...,...,...,...,...,...,...,...
4966,4967,Theo Kalomirakis' Private Theaters,"Bijou, Paramount, Loew's, Fox -- the names the...",Brett Anderson Theo Kalomirakis,http://books.google.com/books/content?id=jytUA...,http://books.google.com/books?id=jytUAAAAMAAJ&...,http://books.google.com/books?id=jytUAAAAMAAJ&...,Architecture,theo kalomirakis' private theaters brett ander...
4967,4968,Comprehensive Care of Schizophrenia,The second edition of this popular volume has ...,Jeffrey A. Lieberman Robin M. Murray,http://books.google.com/books/content?id=opYmp...,http://books.google.nl/books?id=opYmpaeAGxEC&p...,http://books.google.nl/books?id=opYmpaeAGxEC&d...,Medical,comprehensive care of schizophrenia jeffrey a....
4968,4969,"The Handbook of Economic Sociology, Second Edi...",The 37th World Congress of the IIS focused on ...,Peter Hedström Björn Wittrock,http://books.google.com/books/content?id=51lm9...,http://books.google.nl/books?id=51lm99p7PfEC&p...,http://books.google.nl/books?id=51lm99p7PfEC&d...,Social Science,"the handbook of economic sociology, second edi..."
4969,4970,"DB2 Cluster Certification Guide, The",DB2 is a database that is geared for use on th...,Jonathan Cook Calene Janacek Dwaine Snow,http://books.google.com/books/content?id=Fpc3G...,http://books.google.com/books?id=Fpc3GuBTYu4C&...,http://books.google.com/books?id=Fpc3GuBTYu4C&...,Computers,"db2 cluster certification guide, the jonathan ..."


In [64]:
def get_recommendations(book_id, cosine_sim=cosine_sim, df=df):
    if book_id not in df['book_id'].values:
        return {"error": "Book not found"}
        
    # the index of the book that matches the book_id
    idx = df.index[df['book_id'] == book_id].tolist()[0]
    
    # the similarity scores with all books
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # sort them by similarity score in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    sim_scores = sim_scores[1:4]
    
    # prepare the response (JSON Format)
    results = []
    for i in sim_scores:
        results.append({
            "recommended_book_id": int(df['book_id'].iloc[i[0]]),
            "title": df['title'].iloc[i[0]],
            "match_score": f"{round(i[1] * 100, 1)}%"
        })
        
    return results

# test the recommendation system with a sample book_id
test_book_id = 1545
book_name = df[df['book_id'] == test_book_id]['title'].values[0]

print(f" books similar to {book_name} (ID: {test_book_id})")
print("=" * 60)
print(get_recommendations(test_book_id))

 books similar to Empires (ID: 1545)
[{'recommended_book_id': 4356, 'title': 'Who Rules America?: Power and Politics in the Year 2000', 'match_score': '58.9%'}, {'recommended_book_id': 3594, 'title': 'Who Rules Britain?', 'match_score': '56.3%'}, {'recommended_book_id': 3538, 'title': 'The Revolt of the Elites and the Betrayal of Democracy', 'match_score': '55.8%'}]


In [65]:
def get_smart_content_recommendations(user_id, user_department, df_ratings=df_ratings, current_df=df, current_sim_matrix=cosine_sim, num_recommendations=5):
    """
    Smart Content-Based Recommender: 
    Relies purely on NLP Semantic Similarity and User Demographics.
    """
    final_response = {
        "user_id": user_id,
        "department": user_department,
        "status": "success",
        "recommended_picks": [],
        "message": ""
    }
    
    # Map university departments to standard book genres
    department_to_genre = {
        "AI": "Computers",
        "Computer Science": "Computers",
        "Civil": "Technology",
        "Mechanical": "Engineering",
        "Business": "Business",
        "Arts": "Art",
        "Philosophy": "Philosophy"
    }
    
    user_history = df_ratings[df_ratings['user_id'] == user_id]
    books_read = user_history['book_id'].tolist() if not user_history.empty else []
    
    favorite_books = user_history[user_history['rating'] >= 4.0].sort_values(by='rating', ascending=False)
    
    # ==========================================
    # 1. Active User Scenario
    # ==========================================
    if not favorite_books.empty:
        final_response["message"] = "Active User: Generating Content-Based recommendations from top-rated past reads."
        
        top_seed_books = favorite_books.head(2)['book_id'].tolist()
        raw_recommendations = []
        
        for seed_book_id in top_seed_books:
            if seed_book_id in current_df['book_id'].values:
                idx = current_df.index[current_df['book_id'] == seed_book_id].tolist()[0]
                sim_scores = list(enumerate(current_sim_matrix[idx]))
                sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
                
                # Fetch top 10 similar books to filter out already read ones
                for i in sim_scores[1:11]:
                    rec_book_id = int(current_df['book_id'].iloc[i[0]])
                    
                    if rec_book_id not in books_read:
                        raw_recommendations.append({
                            "recommended_book_id": rec_book_id,
                            "title": current_df['title'].iloc[i[0]],
                            "genre": current_df['genre'].iloc[i[0]],
                            "match_score_raw": float(i[1]),
                            "match_score": f"{round(i[1] * 100, 1)}%"
                        })
        
        # Remove duplicates and sort by highest similarity
        unique_recs = {}
        for rec in raw_recommendations:
            b_id = rec['recommended_book_id']
            if b_id not in unique_recs or rec['match_score_raw'] > unique_recs[b_id]['match_score_raw']:
                unique_recs[b_id] = rec
                
        sorted_recs = sorted(list(unique_recs.values()), key=lambda x: x['match_score_raw'], reverse=True)
        
        for rec in sorted_recs:
            del rec['match_score_raw']
            
        final_response["recommended_picks"] = sorted_recs[:num_recommendations]
        return final_response

    # ==========================================
    # 2. Cold Start Scenario
    # ==========================================
    final_response["message"] = f"Cold Start: Recommending foundation books based on department ({user_department})."
    
    search_genre = department_to_genre.get(user_department, user_department)
    department_books = current_df[current_df['genre'].str.contains(search_genre, case=False, na=False)]
    
    if not department_books.empty:
        default_book_id = int(department_books.iloc[0]['book_id'])
    else:
        default_book_id = int(current_df.iloc[0]['book_id']) 
        
    idx = current_df.index[current_df['book_id'] == default_book_id].tolist()[0]
    sim_scores = list(enumerate(current_sim_matrix[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[0:num_recommendations]
    
    dept_recs = []
    for i in sim_scores:
        dept_recs.append({
            "recommended_book_id": int(current_df['book_id'].iloc[i[0]]),
            "title": current_df['title'].iloc[i[0]],
            "genre": current_df['genre'].iloc[i[0]],
            "match_score": f"{round(i[1] * 100, 1)}%"
        })
        
    final_response["recommended_picks"] = dept_recs
    return final_response

In [66]:
test_user_id = 95
test_user_dept = "Business"

print(f"📚 Top rated books previously read by User {test_user_id}:")
print("-" * 80)

user_history = df_ratings[df_ratings['user_id'] == test_user_id].sort_values(by='rating', ascending=False)

if user_history.empty:
    print("No past history found for this user (Cold Start).")
else:
    history_details = pd.merge(user_history, df[['book_id', 'title', 'genre']], on='book_id', how='left')
    for _, row in history_details.head(5).iterrows():
        print(f"⭐ Actual Rating: {row['rating']}/5.0 | {row['title']}")
        print(f"   Genre: [{row['genre']}]\n")

print("=" * 80)

print(f"✨ AI Smart Content-Based Recommendations for User {test_user_id}:")
print("-" * 80)

results = get_smart_content_recommendations(
    user_id=test_user_id, 
    user_department=test_user_dept,
    df_ratings=df_ratings,
    current_df=df,
    current_sim_matrix=cosine_sim,
    num_recommendations=5
)

print(f"📌 Logic Used: {results['message']}\n")

if not results['recommended_picks']:
    print("No recommendations available.")
else:
    for rec in results['recommended_picks']:
        print(f"🎯 Match Score: {rec['match_score']} | {rec['title']}")
        print(f"   Genre: [{rec['genre']}]\n")

📚 Top rated books previously read by User 95:
--------------------------------------------------------------------------------
⭐ Actual Rating: 5.0/5.0 | Crucial Confrontations
   Genre: [Business & Economics]

⭐ Actual Rating: 5.0/5.0 | Intrinsic Motivation at Work: Building Energy and Commitment
   Genre: [Business & Economics]

⭐ Actual Rating: 5.0/5.0 | Breakthrough: Stories and Strategies of Radical Innovation
   Genre: [Business & Economics]

⭐ Actual Rating: 5.0/5.0 | Happiness: Lessons from a New Science
   Genre: [Psychology]

⭐ Actual Rating: 5.0/5.0 | Bare-Knuckle Negotiation: Savvy Tips and True Stories from the Master of Give-and-Take
   Genre: [Business & Economics]

✨ AI Smart Content-Based Recommendations for User 95:
--------------------------------------------------------------------------------
📌 Logic Used: Active User: Generating Content-Based recommendations from top-rated past reads.

🎯 Match Score: 71.9% | The Dynamic Society: The Sources of Global Change
   Gen

## **NLP Chatpot:**

In [67]:
def chatbot_semantic_search(user_message, nlp_model, df_books, corpus_embeddings, top_k=5, nlp_threshold=0.40):
    """
    Pure Semantic Search: Recommends books based ONLY on the user's question, ignoring past history.
    """
    # 1. Embed the user's question
    query_embedding = nlp_model.encode(user_message, convert_to_tensor=True)
    
    # 2. Calculate similarities with all books
    cosine_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
    
    # 3. Sort and get top results
    top_results = np.argsort(cosine_scores.cpu().numpy())[::-1][:top_k * 2]
    
    final_picks = []
    for idx in top_results:
        score = float(cosine_scores[idx])
        # Only accept books that have a solid semantic match
        if score >= nlp_threshold:
            final_picks.append({
                "recommended_book_id": int(df_books.iloc[idx]['book_id']),
                "title": df_books.iloc[idx]['title'],
                "match_score": round(score * 100, 1)
            })
            
    # Keep only the requested number of books (top_k)
    final_picks = final_picks[:top_k]
            
    if not final_picks:
        return {
            "bot_reply": "عذراً، لم أتمكن من العثور على كتب تطابق طلبك بدقة في مكتبتنا حالياً. هل يمكنك توضيح بحثك أو استخدام مصطلحات أخرى؟",
            "recommended_books": []
        }

    return {
        "bot_reply": "", 
        "recommended_books": final_picks
    }

In [68]:
load_dotenv(override=True)
API_KEY = os.getenv("GEMINI_API_KEY")

if not API_KEY:
    raise ValueError("GEMINI_API_KEY is missing! Please check your .env file.")

genai.configure(api_key=API_KEY)

llm_model = genai.GenerativeModel('gemini-3.1-flash-lite-preview')

def generate_rag_response(user_message, search_results):
    """
    Simplified RAG Engine with Silent Veto Power: Answers strictly based on current question, 
    silently drops irrelevant search results without meta-commentary, and uses neutral greetings.
    """
    if not search_results:
        return "عذراً، لم أجد كتباً تطابق طلبك بدقة. الفهرس الحالي قد لا يحتوي على مراجع في هذا التخصص الدقيق."
    
    context_text = ""
    for book in search_results:
        context_text += f"- Book Title: [{book['title']}]\n"
        context_text += f"  Match Score: {book['match_score']}%\n"

    # The Strictly Controlled Prompt
    prompt = f"""
    أنت مستشار أكاديمي خبير وموجه للطلاب في مكتبة الجامعة. دورك هو إرشاد الطالب للإجابة على سؤاله بناءً على نتائج البحث.
    طريقتك رسمية، مهنية، أكاديمية، وحيادية تماماً باللغة العربية الفصحى.
    يجب الاحتفاظ بالمصطلحات التقنية باللغة الإنجليزية كما هي بين أقواس مربعة [ ].

    استفسار الطالب: "{user_message}"
    
    نتائج البحث من فهرس المكتبة:
    {context_text}
    
    المطلوب منك بالترتيب:
    1. تحية رسمية محايدة: (يُمنع منعاً باتاً استخدام ألقاب مثل: يا بني، عزيزي، أيها الطالب، إلخ).
    2. التصفية الذكية الصامتة (Silent Veto): راجع الكتب المقترحة أولاً. إذا وجدت كتاباً ليس له علاقة منطقية بمجال الطالب، تجاهله تماماً. **(قانون صارم: يُمنع إخبار الطالب أنك قمت باستبعاد كتب، ويُمنع ذكر أسماء الكتب المستبعدة. الفلترة يجب أن تحدث في صمت تام دون تبرير)**.
    3. التقييم الأكاديمي: للكتب التي نجت من التصفية (إن وجدت)، اعرضها في نقاط (Bullet points) واشرح بصدق وموضوعية كيف تفيده. 
    4. التعامل مع النقص: إذا قمت بتجاهل كل النتائج لأنها غير مناسبة، صارح الطالب بلباقة أن المكتبة تفتقر حالياً للمراجع المتخصصة في مجاله الدقيق، واقترح عليه مصطلحات بحث بديلة.
    5. اختم رسالتك بعبارة ختامية مهنية قصيرة (مثل: "مع خالص التمنيات بالتوفيق"، أو ما شابه).
    
    قيود صارمة (RAG Constraints):
    - لا تقترح أي كتاب من خارج (نتائج البحث المطابقة) إطلاقاً.
    - الفلترة صامتة: لا تكتب أي جملة تفيد بأنك قمت بمراجعة أو تصفية أو استبعاد نتائج.
    - خلو النص تماماً من أي رموز تعبيرية (Emojis).
    """
    
    try:
        response = llm_model.generate_content(prompt)
        return response.text
    except Exception as e:
        return f"حدث خطأ في الاتصال: {e}"

In [69]:
current_question = "عندك كتب تساعدني في فهم ال probability"
# 1. Search based ONLY on the question
search_results = chatbot_semantic_search(
    user_message=current_question, 
    nlp_model=nlp_model, 
    df_books=df, 
    corpus_embeddings=book_embeddings
)

actual_books = search_results.get("recommended_books", [])

# 2. Generate the AI response
print("Generating AI response...\n" + "=" * 60)
final_reply = generate_rag_response(current_question, actual_books)
print(final_reply)

print("\n--- Internal JSON Data for Backend ---")
print(json.dumps(actual_books, indent=4, ensure_ascii=False))

Generating AI response...
أهلاً بك.

بناءً على استفسارك حول المصادر المتاحة لفهم نظرية الاحتمالات [Probability Theory]، تتوفر في مكتبة الجامعة المراجع التالية التي تغطي جوانب متعددة من هذا التخصص:

*   [Foundations of Probability with Applications : Selected Papers 1974-1995 (Cambridge Studies in Probability, Induction and Decision Theory)]: يعد هذا المرجع مناسباً للباحثين الراغبين في الاطلاع على الجوانب التطبيقية والأسس النظرية المتقدمة التي تمت صياغتها خلال فترة زمنية محددة، وهو مفيد لتعميق الفهم الاستدلالي.
*   [A Course in Probability Theory Revised]: يعتبر مصدراً أكاديمياً رصيناً يقدم منهجية تعليمية شاملة، ويصلح للطلاب الذين يسعون لبناء قاعدة معرفية متينة ومنهجية في [Probability Theory].
*   [Probability Models (Springer Undergraduate Mathematics Series)]: يركز هذا الكتاب على النماذج الاحتمالية، وهو مناسب جداً للمستوى الجامعي؛ حيث يربط بين المفاهيم الرياضية وتطبيقاتها العملية في سياق أكاديمي مبسط.
*   [Elementary Probability Theory]: يقدم هذا المرجع المفاهيم الأساسية والمبادئ الأو

## **Save Models:**

In [70]:
# 1. Create the production directory
MODEL_DIR = 'production_models'
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Exporting AI artifacts to '{MODEL_DIR}/' ...\n" + "-"*60)

# ==========================================
# Phase 1: Core Data & Content-Based Matrices
# ==========================================
joblib.dump(df_ratings, f'{MODEL_DIR}/df_ratings.pkl')
print("✅ Saved Ratings History: df_ratings.pkl")

joblib.dump(df, f'{MODEL_DIR}/books_dataframe.pkl')
joblib.dump(cosine_sim, f'{MODEL_DIR}/content_sim_matrix.pkl')
print("✅ Saved Content-Based Matrices: books_dataframe.pkl, content_sim_matrix.pkl")

# ==========================================
# Phase 2: NLP & Semantic Search Artifacts
# ==========================================
if 'book_embeddings' in locals() or 'book_embeddings' in globals():
    if isinstance(book_embeddings, torch.Tensor):
        embeddings_np = book_embeddings.cpu().numpy()
    else:
        embeddings_np = np.array(book_embeddings)
        
    joblib.dump(embeddings_np, f'{MODEL_DIR}/nlp_book_embeddings.pkl')
    joblib.dump(df, f'{MODEL_DIR}/nlp_books_dataframe.pkl')
    print("✅ Saved NLP Embeddings: nlp_book_embeddings.pkl, nlp_books_dataframe.pkl")

print("-" * 60)
print("🚀 All files successfully exported!")

Exporting AI artifacts to 'production_models/' ...
------------------------------------------------------------
✅ Saved Ratings History: df_ratings.pkl
✅ Saved Content-Based Matrices: books_dataframe.pkl, content_sim_matrix.pkl
✅ Saved NLP Embeddings: nlp_book_embeddings.pkl, nlp_books_dataframe.pkl
------------------------------------------------------------
🚀 All files successfully exported!
